# MOF-74 (Mn) Linker — H₄DOBDC Electronic Properties

This notebook builds the **2,5-dihydroxyterephthalic acid (H₄DOBDC)** organic linker
of **MOF-74-Mn (CPO-27-Mn)** from SMILES and computes its electronic properties.

```
SMILES ──► RDKit ETKDG ──► ASE Atoms ──► GOAL AtomicGraph ──► MACE (forces/energy)
                │
                ▼
           xTB GFN2 ──► HOMO / LUMO / Gap / Conceptual-DFT descriptors
                │
                ▼
           ORCA B3LYP ──► High-quality orbital energies (optional, needs ORCA binary)
```

## Ligand: H₄DOBDC

| Property | Value |
|---|---|
| Full name | 2,5-dihydroxyterephthalic acid |
| Formula | C₈H₆O₆ (neutral acid, 4 ionisable protons) |
| MW | 198.13 g/mol |
| CAS | 610-92-4 |
| In MOF-74 | Deprotonated as DOBDC⁴⁻; carboxylate + phenolate chelate each Mn |

All calculations in this notebook use the **neutral acid** (charge = 0, singlet).
Swap `CHARGE = -2` or `-4` in the config cell to study deprotonated forms.

---
## 0 · Configuration

In [ ]:
# ── Molecule ─────────────────────────────────────────────────────────────────
# Canonical SMILES for 2,5-dihydroxyterephthalic acid (H₄DOBDC, neutral acid)
# CAS 610-92-4 / PubChem CID: 73699
SMILES = "OC(=O)c1cc(O)c(C(=O)O)cc1O"
CHARGE = 0        # 0 = neutral acid; -2 = dianion; -4 = tetraanion (MOF-74 form)
UHF    = 0        # unpaired electrons (0 = singlet, closed-shell)
MOLECULE_NAME = "H4DOBDC"

# ── Neighbour-list cutoff for GOAL AtomicGraph ───────────────────────────────
CUTOFF = 5.0   # Å

# ── ORCA setup (set to None to skip the ORCA section) ───────────────────────
# Example: ORCA_CMD = "/opt/orca/orca"  or  ORCA_CMD = shutil.which("orca")
# Leave as None if ORCA is not installed — xTB sections still run without it.
ORCA_CMD       = None          # ← SET YOUR ORCA PATH HERE
ORCA_KEYWORDS  = "! B3LYP def2-SVP D3BJ TightSCF"  # DFT level
ORCA_XTB2_KEYWORDS = "! XTB2"                       # fast ORCA-xTB2 alternative
USE_XTB2_IN_ORCA   = False     # True → use !XTB2 instead of B3LYP

# ── MACE ─────────────────────────────────────────────────────────────────────
MACE_MODEL  = "small"   # "small" | "medium" | "large"  (downloads ~300 MB on first run)
MACE_DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"

# ── Output directory ─────────────────────────────────────────────────────────
from pathlib import Path
WORK_DIR = Path("dobdc_work")
WORK_DIR.mkdir(exist_ok=True)

print(f"Molecule     : {MOLECULE_NAME}  (charge={CHARGE}, uhf={UHF})")
print(f"SMILES       : {SMILES}")
print(f"Cutoff       : {CUTOFF} Å")
print(f"ORCA         : {'set → ' + ORCA_CMD if ORCA_CMD else 'not configured (skipping ORCA section)'}")
print(f"MACE device  : {MACE_DEVICE}")

---
## 1 · Imports

In [ ]:
from __future__ import annotations
import io as sio
import re
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── ASE ───────────────────────────────────────────────────────────────────────
import ase
from ase import Atoms
from ase.io import write as ase_write, read as ase_read
from ase.neighborlist import natural_cutoffs, NeighborList

# ── GOAL ──────────────────────────────────────────────────────────────────────
import goal.ml.cli  # noqa: F401 — populates backbone/head registries
from goal.ml.data.graph import AtomicGraph

# ── RDKit ─────────────────────────────────────────────────────────────────────
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw, rdMolDescriptors
    from rdkit.Chem.Draw import rdMolDraw2D
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False
    print("⚠  RDKit not found. Run with the 'md' pixi env or: pip install rdkit")

# ── py3Dmol ───────────────────────────────────────────────────────────────────
try:
    import py3Dmol
    HAS_PY3DMOL = True
except ImportError:
    HAS_PY3DMOL = False

# ── xTB ───────────────────────────────────────────────────────────────────────
try:
    from xtb.ase.calculator import XTB as XTBCalcASE
    HAS_XTB_ASE = True
except ImportError:
    HAS_XTB_ASE = False

try:
    from xtb.interface import Calculator as XTBInterface, Param
    from xtb.libxtb import VERBOSITY_MUTED
    HAS_XTB_IFACE = True
except ImportError:
    HAS_XTB_IFACE = False

if not HAS_XTB_ASE:
    print("⚠  xtb-python not found. Run with the 'md' pixi env or: pip install xtb-python")

# ── MACE ──────────────────────────────────────────────────────────────────────
try:
    from mace.calculators import mace_mp
    HAS_MACE = True
except ImportError:
    HAS_MACE = False
    print("⚠  mace-torch not found. Install with: pip install mace-torch")

# ── ORCA ASE interface ────────────────────────────────────────────────────────
try:
    from ase.calculators.orca import ORCA, OrcaProfile, OrcaTemplate
    HAS_ORCA_ASE = True
except ImportError:
    HAS_ORCA_ASE = False

print(f"PyTorch  : {torch.__version__}")
print(f"ASE      : {ase.__version__}")
print(f"RDKit    : {'✓' if HAS_RDKIT else '✗'}")
print(f"xTB-ASE  : {'✓' if HAS_XTB_ASE else '✗'}")
print(f"xTB-API  : {'✓' if HAS_XTB_IFACE else '✗'}")
print(f"MACE     : {'✓' if HAS_MACE else '✗'}")
print(f"py3Dmol  : {'✓' if HAS_PY3DMOL else '✗'}")
print(f"ORCA-ASE : {'✓' if HAS_ORCA_ASE else '✗'}")

---
## 2 · Build the Ligand Geometry

Starting from SMILES we:
1. Parse with RDKit and add explicit hydrogens
2. Generate a 3D conformation with the ETKDG distance-geometry algorithm
3. Minimise with the MMFF94 force field (fast, chemically reasonable geometry)
4. Wrap in an ASE `Atoms` object for downstream use

This step is purely classical — no QM yet.

In [ ]:
assert HAS_RDKIT, "RDKit is required for this cell."

# ── Parse SMILES ──────────────────────────────────────────────────────────────
mol = Chem.MolFromSmiles(SMILES)
assert mol is not None, f"RDKit could not parse SMILES: {SMILES}"

# Add implicit hydrogens (important — H₄DOBDC has 4 OH/COOH protons)
mol = Chem.AddHs(mol)

# ── 3D conformation: ETKDG ───────────────────────────────────────────────────
params = AllChem.ETKDGv3()
params.randomSeed = 42
status = AllChem.EmbedMolecule(mol, params)
assert status == 0, "ETKDG conformation generation failed."

# ── MMFF94 optimisation ───────────────────────────────────────────────────────
ff_result = AllChem.MMFFOptimizeMolecule(mol, mmffVariant="MMFF94", maxIters=2000)
if ff_result != 0:
    print("⚠  MMFF94 did not fully converge (code={ff_result}), using current geometry.")

# ── Build ASE Atoms ───────────────────────────────────────────────────────────
conformer = mol.GetConformer(0)
positions = np.array(conformer.GetPositions())   # Ångström
symbols   = [atom.GetSymbol() for atom in mol.GetAtoms()]

atoms = Atoms(symbols=symbols, positions=positions)
# No periodic boundary conditions for an isolated molecule
atoms.set_pbc(False)

# Save initial geometry
ase_write(str(WORK_DIR / "dobdc_mmff94.xyz"), atoms)

# ── Molecular formula check ────────────────────────────────────────────────────
rdkit_formula = rdMolDescriptors.CalcMolFormula(mol)
print(f"RDKit formula : {rdkit_formula}")
print(f"Expected      : C8H6O6")
print(f"Atoms         : {atoms.get_chemical_formula()}  ({len(atoms)} total)")
print(f"Atom types    : {sorted(set(symbols))}")
print(f"Positions     : {positions.shape}")
print()
print("First 5 atoms (symbol, x, y, z in Å):")
for sym, pos in zip(symbols[:5], positions[:5]):
    print(f"  {sym:2s}  {pos[0]:8.4f}  {pos[1]:8.4f}  {pos[2]:8.4f}")

### 2.1 · 3D Structure Preview

A quick look at the MMFF94 geometry straight after construction.
Colours follow CPK convention: **C = dark grey · O = red · H = light grey**.
Rotate with left-drag, zoom with scroll, right-drag to translate.

In [ ]:
if not HAS_PY3DMOL:
    print("py3Dmol not installed — skipping 3D preview.")
    print("Install with: pip install py3Dmol")
else:
    import io as _io
    from ase.io import write as _aw
    from ase.neighborlist import natural_cutoffs as _ncut, NeighborList as _NL

    # ── Serialise to XYZ ──────────────────────────────────────────────────────
    _buf = _io.StringIO()
    _aw(_buf, atoms, format="xyz")
    _xyz = _buf.getvalue()

    # CPK palette and sphere radii (Å, ~30% of van-der-Waals)
    _COLORS = {
        "H": "#e8e8e8", "C": "#404040", "O": "#cc2200",
        "N": "#3355ff", "S": "#ddcc00", "F": "#22aa44",
    }
    _RADII = {"H": 0.13, "C": 0.22, "O": 0.22, "N": 0.22, "S": 0.26, "F": 0.20}

    _syms = atoms.get_chemical_symbols()
    _pos  = atoms.get_positions()

    view = py3Dmol.view(width=780, height=520)
    view.addModel(_xyz, "xyz")
    view.setStyle({}, {})   # clear py3Dmol defaults

    # ── Covalent bonds via ASE neighbour list (natural cutoffs + 10 %) ────────
    _cuts = _ncut(atoms, mult=1.1)
    _nl   = _NL(_cuts, self_interaction=False, bothways=False)
    _nl.update(atoms)
    for _i in range(len(atoms)):
        _nbrs, _ = _nl.get_neighbors(_i)
        for _j in _nbrs:
            if _j > _i:
                _p1, _p2 = _pos[_i], _pos[_j]
                view.addCylinder({
                    "start":   {"x": float(_p1[0]), "y": float(_p1[1]), "z": float(_p1[2])},
                    "end":     {"x": float(_p2[0]), "y": float(_p2[1]), "z": float(_p2[2])},
                    "radius":  0.075,
                    "color":   "#999999",
                    "fromCap": True,
                    "toCap":   True,
                })

    # ── Atom spheres ──────────────────────────────────────────────────────────
    for _sym, _p in zip(_syms, _pos):
        view.addSphere({
            "center": {"x": float(_p[0]), "y": float(_p[1]), "z": float(_p[2])},
            "radius": _RADII.get(_sym, 0.20),
            "color":  _COLORS.get(_sym, "#cccccc"),
        })

    # ── Labels on heavy atoms (element + index) ───────────────────────────────
    for _i, (_sym, _p) in enumerate(zip(_syms, _pos)):
        if _sym != "H":
            view.addLabel(
                f"{_sym}{_i}",
                {
                    "position":       {"x": float(_p[0]), "y": float(_p[1]), "z": float(_p[2])},
                    "fontSize":       11,
                    "fontColor":      "white",
                    "showBackground": False,
                    "alignment":      "center",
                },
            )

    view.setBackgroundColor("#1a1a2e")  # dark background makes colours pop
    view.zoomTo()
    view.show()

    # Legend
    _elem_counts = {}
    for _s in _syms:
        _elem_counts[_s] = _elem_counts.get(_s, 0) + 1
    print(f"H₄DOBDC  —  {atoms.get_chemical_formula()}  ({len(atoms)} atoms)")
    print("Composition : " + "  ".join(f"{k}:{v}" for k, v in sorted(_elem_counts.items())))
    print("Geometry    : SMILES → RDKit ETKDGv3 + MMFF94 minimisation")

### 2.2 · 2D Structure (RDKit)

In [ ]:
assert HAS_RDKIT, "RDKit required."

from IPython.display import Image, display

# Generate 2D depiction coordinates
mol2d = Chem.MolFromSmiles(SMILES)
AllChem.Compute2DCoords(mol2d)

drawer = rdMolDraw2D.MolDraw2DCairo(400, 300)
drawer.drawOptions().addAtomIndices = False
drawer.drawOptions().addStereoAnnotation = False
drawer.DrawMolecule(mol2d)
drawer.FinishDrawing()

img_path = WORK_DIR / "dobdc_2d.png"
with open(img_path, "wb") as f:
    f.write(drawer.GetDrawingText())
display(Image(str(img_path)))
print("2D structure: H₄DOBDC (2,5-dihydroxyterephthalic acid)")
print("- Two COOH groups at positions 1 and 4")
print("- Two OH groups at positions 2 and 5  (ortho to each COOH)")
print("- Each (COOH + OH) pair provides bidentate coordination to Mn in MOF-74")

---
## 3 · 3D Visualisation (py3Dmol)

Atoms are coloured by element: C = grey, O = red, H = white.  
Bonds are drawn using ASE natural cutoffs (covalent radii).

In [ ]:
# Element colours (CPK)
_CPK = {"H": "#ffffff", "C": "#555555", "O": "#cc2200", "N": "#3355ff"}

def ase_to_xyz_string(at: Atoms) -> str:
    buf = sio.StringIO()
    ase_write(buf, at, format="xyz")
    return buf.getvalue()

def show_molecule(at: Atoms, width: int = 700, height: int = 450) -> None:
    """Render ASE Atoms as a 3D stick model in py3Dmol."""
    if not HAS_PY3DMOL:
        print("py3Dmol not installed — skipping 3D view.")
        return

    syms = at.get_chemical_symbols()
    pos  = at.get_positions()
    xyz  = ase_to_xyz_string(at)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(xyz, "xyz")
    view.setStyle({}, {})  # suppress py3Dmol default spheres

    # Bonds via ASE natural cutoffs
    cutoffs = natural_cutoffs(at, mult=1.1)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=False)
    nl.update(at)
    for i in range(len(at)):
        indices, _ = nl.get_neighbors(i)
        for j in indices:
            if j > i:
                p1, p2 = pos[i], pos[j]
                view.addCylinder({
                    "start":  {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                    "end":    {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                    "radius": 0.07, "color": "#888888",
                    "fromCap": True, "toCap": True,
                })

    # Atom spheres
    for sym, p in zip(syms, pos):
        r = 0.14 if sym == "H" else 0.22
        view.addSphere({
            "center": {"x": float(p[0]), "y": float(p[1]), "z": float(p[2])},
            "radius": r,
            "color":  _CPK.get(sym, "#cccccc"),
        })

    view.zoomTo()
    view.show()

show_molecule(atoms)
print(f"H₄DOBDC after MMFF94 optimisation: {len(atoms)} atoms")

---
## 4 · GOAL AtomicGraph Construction

`AtomicGraph.from_ase()` is the canonical entry point.  It:
1. Converts positions and atomic numbers to `torch.Tensor`
2. Builds a neighbour list with the given cutoff (ASE backend, correct MIC)
3. Returns a PyTorch Geometric `Data` subclass ready for any GOAL model

> **Note:** `edge_vectors` and `edge_lengths` are *not* stored on the graph —
> production force heads recompute them in the forward pass so autograd can
> flow through positions.  See [graph.py](../src/goal/ml/data/graph.py) for details.

In [ ]:
graph: AtomicGraph = AtomicGraph.from_ase(
    atoms,
    cutoff=CUTOFF,
    dtype=torch.float64,
)

print("═══ AtomicGraph ═══════════════════════════════════════")
print(f"  Molecule          : {atoms.get_chemical_formula()}")
print(f"  Number of atoms N : {graph.num_atoms}")
print(f"  pos (positions)   : {tuple(graph.pos.shape)}   float64")
print(f"  z   (atomic nums) : {tuple(graph.z.shape)}   int64")
print(f"  edge_index        : {tuple(graph.edge_index.shape)}   int64")
print(f"  unit_shifts       : {tuple(graph.unit_shifts.shape)}   int64  (all zeros, no PBC)")
print(f"  cell              : {tuple(graph.cell.shape)}   (zeros, aperiodic)")
print(f"  pbc               : {graph.pbc.tolist()}")
print()
print(f"  Cutoff            : {CUTOFF} Å")
print(f"  Number of edges E : {graph.edge_index.shape[1]}")
print(f"  Avg neighbours    : {graph.edge_index.shape[1] / graph.num_atoms:.2f} per atom")
print()
print("─── Atomic numbers ──────────────────────────────────")
from collections import Counter
import ase.data
z_counts = Counter(graph.z.tolist())
for z, count in sorted(z_counts.items()):
    print(f"  Z={z:3d}  ({ase.data.chemical_symbols[z]})  : {count} atoms")

In [ ]:
# ── Edge length distribution ──────────────────────────────────────────────────
# Recompute edge vectors manually (same as the production heads do in forward)
src, dst = graph.edge_index
edge_vecs   = graph.pos[dst] - graph.pos[src]    # r_j - r_i
edge_lengths = edge_vecs.norm(dim=-1)             # ‖r_j - r_i‖

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(edge_lengths.numpy(), bins=30, color="#2196F3", edgecolor="white", linewidth=0.5)
ax.set_xlabel("Edge length (Å)")
ax.set_ylabel("Count")
ax.set_title(f"H₄DOBDC — Edge length distribution (cutoff = {CUTOFF} Å)")
ax.axvline(edge_lengths.min().item(), color="red",   ls="--", label=f"min = {edge_lengths.min():.2f} Å")
ax.axvline(edge_lengths.max().item(), color="green", ls="--", label=f"max = {edge_lengths.max():.2f} Å")
ax.legend()
fig.tight_layout()
plt.show()
print(f"Edge lengths — min: {edge_lengths.min():.3f} Å, max: {edge_lengths.max():.3f} Å, "
      f"mean: {edge_lengths.mean():.3f} Å")

---
## 5 · MACE Pre-trained Model — Energy & Forces

We use **MACE-MP-0** (Materials Project foundation model) as a stand-in until a
SIMURGH model trained on MOF-relevant data is available.  MACE-MP covers all
elements H–Rn and is appropriate for organic ligands.

> **Swap to SIMURGH later:** replace `calc_mace` with
> `GOALCalculator(checkpoint_path="<your_checkpoint>")` once you train a
> universal SIMURGH.  The `atoms.calc = ...` interface is identical.

MACE-MP weights (~300 MB) are downloaded automatically on first run.

In [ ]:
if not HAS_MACE:
    print("MACE not installed — skipping this section.")
    print("Install with: pip install mace-torch")
else:
    print(f"Loading MACE-MP-0 ({MACE_MODEL}) on {MACE_DEVICE} …")
    calc_mace = mace_mp(model=MACE_MODEL, device=MACE_DEVICE, default_dtype="float64")
    print("Loaded.")

In [ ]:
if not HAS_MACE:
    print("MACE not available — skipping.")
else:
    atoms_mace = atoms.copy()
    atoms_mace.calc = calc_mace

    energy_mace = atoms_mace.get_potential_energy()   # eV
    forces_mace = atoms_mace.get_forces()             # eV/Å, shape (N, 3)

    f_norms = np.linalg.norm(forces_mace, axis=1)    # per-atom force magnitude

    print("═══ MACE-MP-0 Results ══════════════════════════════")
    print(f"  Energy              : {energy_mace:.6f} eV")
    print(f"  Energy / atom       : {energy_mace / len(atoms_mace):.6f} eV/atom")
    print(f"  Max force magnitude : {f_norms.max():.4f} eV/Å")
    print(f"  Mean force magnitude: {f_norms.mean():.4f} eV/Å")
    print(f"  Net force (Newton)  : [{forces_mace.sum(axis=0)[0]:+.4e}  "
          f"{forces_mace.sum(axis=0)[1]:+.4e}  "
          f"{forces_mace.sum(axis=0)[2]:+.4e}] eV/Å")
    print()
    print("Forces per atom (eV/Å):")
    syms = atoms_mace.get_chemical_symbols()
    for i, (sym, f, fn) in enumerate(zip(syms, forces_mace, f_norms)):
        print(f"  {i:2d}  {sym:2s}  |F|={fn:.4f}  "
              f"[{f[0]:+.4f}  {f[1]:+.4f}  {f[2]:+.4f}]")

In [ ]:
if HAS_MACE and HAS_PY3DMOL:
    # Colour atoms by MACE force magnitude (white → red)
    f_norms_norm = (f_norms - f_norms.min()) / (f_norms.max() - f_norms.min() + 1e-10)
    cmap_f = plt.get_cmap("hot_r")

    xyz_str = ase_to_xyz_string(atoms_mace)
    pos = atoms_mace.get_positions()
    view = py3Dmol.view(width=700, height=450)
    view.addModel(xyz_str, "xyz")
    view.setStyle({}, {})

    cutoffs = natural_cutoffs(atoms_mace, mult=1.1)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=False)
    nl.update(atoms_mace)
    for i in range(len(atoms_mace)):
        indices, _ = nl.get_neighbors(i)
        for j in indices:
            if j > i:
                p1, p2 = pos[i], pos[j]
                view.addCylinder({
                    "start":  {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                    "end":    {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                    "radius": 0.06, "color": "#888888",
                    "fromCap": True, "toCap": True,
                })

    syms = atoms_mace.get_chemical_symbols()
    for i, (sym, p) in enumerate(zip(syms, pos)):
        rgba = cmap_f(float(f_norms_norm[i]))
        hex_c = mcolors.to_hex(rgba)
        r = 0.13 if sym == "H" else 0.20
        view.addSphere({"center": {"x": float(p[0]), "y": float(p[1]), "z": float(p[2])},
                        "radius": r, "color": hex_c})

    view.zoomTo()
    view.show()
    print("Colour: white (low MACE force) → dark red (high force)")
elif HAS_MACE:
    # Fallback: matplotlib bar chart of force magnitudes
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(len(f_norms)), f_norms, color="steelblue")
    ax.set_xlabel("Atom index")
    ax.set_ylabel("|F| (eV/Å)")
    ax.set_title("MACE-MP-0 force magnitudes on H₄DOBDC")
    fig.tight_layout()
    plt.show()

### 5.2 · Geometry Relaxation with MACE

Relax the MMFF94 geometry on the MACE-MP-0 potential energy surface.  
The relaxed structure is used as input for the QM calculations.

In [ ]:
if not HAS_MACE:
    print("MACE not available — using raw MMFF94 geometry for downstream QM.")
    atoms_qm = atoms.copy()    # MMFF94 geometry
else:
    from ase.optimize import BFGS

    atoms_relax = atoms.copy()
    atoms_relax.calc = calc_mace

    e0  = atoms_relax.get_potential_energy()
    f0m = np.abs(atoms_relax.get_forces()).max()
    print(f"Before relaxation — E = {e0:.4f} eV,  max|F| = {f0m:.4f} eV/Å")

    opt = BFGS(atoms_relax, logfile=None)
    converged = opt.run(fmax=0.05, steps=300)

    ef  = atoms_relax.get_potential_energy()
    ffm = np.abs(atoms_relax.get_forces()).max()
    print(f"After  relaxation — E = {ef:.4f} eV,  max|F| = {ffm:.4f} eV/Å")
    print(f"Steps taken : {opt.nsteps},  converged : {converged}")
    print(f"ΔE (relaxation) : {ef - e0:.4f} eV")

    ase_write(str(WORK_DIR / "dobdc_mace_relaxed.xyz"), atoms_relax)
    atoms_qm = atoms_relax.copy()   # hand relaxed geometry to QM
    print("\nRelaxed geometry saved → dobdc_work/dobdc_mace_relaxed.xyz")

---
## 6 · xTB Electronic Properties (GFN2-xTB)

**GFN2-xTB** is a tight-binding semi-empirical method that gives good HOMO/LUMO
energies for organic molecules at negligible cost.  It is a natural pre-screen
before ORCA DFT.

We use two APIs:
- `xtb.ase.calculator.XTB` — for energy, forces, charges, dipole (ASE-compatible)
- `xtb.interface.Calculator` — for direct access to MO eigenvalues

Orbital energies are in Hartree from the interface; we convert to eV.

Constants:
```
1 Hartree = 27.2114 eV
1 Bohr    = 0.52918 Å
```

In [ ]:
HARTREE_TO_EV = 27.211386245988
BOHR_TO_ANG   = 0.529177210903
ANG_TO_BOHR   = 1.0 / BOHR_TO_ANG

if not HAS_XTB_ASE:
    print("xtb-python not available — skipping xTB section.")
else:
    atoms_xtb = atoms_qm.copy()
    atoms_xtb.calc = XTBCalcASE(
        method="GFN2-xTB",
        charge=CHARGE,
        uhf=UHF,
        accuracy=1.0,
        electronic_temperature=300.0,
    )

    # Single-point (no forces needed yet)
    e_xtb = atoms_xtb.get_potential_energy()    # eV
    f_xtb = atoms_xtb.get_forces()              # eV/Å

    print("═══ GFN2-xTB Single-point ═══════════════════════")
    print(f"  Total energy  : {e_xtb:.6f} eV")
    print(f"  Energy/atom   : {e_xtb / len(atoms_xtb):.6f} eV/atom")
    print(f"  Max |F|       : {np.abs(f_xtb).max():.4f} eV/Å")

    # Charges and dipole (if available in results)
    res_keys = list(atoms_xtb.calc.results.keys())
    print(f"\n  Results available : {res_keys}")

### 6.1 · HOMO / LUMO / Gap via the xTB Python Interface

In [ ]:
if not HAS_XTB_IFACE:
    print("xtb.interface not available — install xtb-python>=22.1")
    xtb_orb_data = None
else:
    # ── Build the low-level xTB calculator ────────────────────────────────────
    numbers   = np.array(atoms_qm.get_atomic_numbers(), dtype=np.int32)
    positions_bohr = atoms_qm.get_positions() * ANG_TO_BOHR   # MUST be Bohr

    # Suppress xTB's verbose banner
    try:
        from xtb.interface import Environment
        env = Environment()
        env.set_verbosity(VERBOSITY_MUTED)
        xtb_iface = XTBInterface(
            Param.GFN2xTB,
            numbers,
            positions_bohr,
            charge=float(CHARGE),
            uhf=UHF,
        )
        xtb_iface.set_verbosity(VERBOSITY_MUTED)
        res_iface = xtb_iface.singlepoint()
    except TypeError:
        # Older API: no environment parameter
        xtb_iface = XTBInterface(
            Param.GFN2xTB,
            numbers,
            positions_bohr,
            charge=float(CHARGE),
            uhf=UHF,
        )
        res_iface = xtb_iface.singlepoint()

    # ── Extract orbital eigenvalues ────────────────────────────────────────────
    # Results object provides eigenvalues in Hartree
    try:
        eig_Ha  = res_iface.get_orbital_eigenvalues()   # shape (n_mo,) or (2, n_mo) for UHF
        occ     = res_iface.get_orbital_occupations()   # 0.0 = virtual, 2.0 = occupied
    except AttributeError:
        # Fallback: older xtb-python uses different names
        eig_Ha  = np.array(res_iface.get_mo_energies())
        occ     = np.array(res_iface.get_mo_occupancies())

    # For restricted (RHF) systems, eig_Ha is 1D; for UHF it may be (2, n_mo)
    if eig_Ha.ndim == 2:
        # alpha spin only for HOMO/LUMO in UHF
        eig_Ha_alpha = eig_Ha[0]
        occ_alpha    = occ[0]
    else:
        eig_Ha_alpha = eig_Ha
        occ_alpha    = occ

    eig_eV = eig_Ha_alpha * HARTREE_TO_EV

    # Identify HOMO (last occ > 0.5) and LUMO (first occ < 0.5)
    occ_mask  = occ_alpha > 0.5
    homo_idx  = np.where(occ_mask)[0][-1]
    lumo_idx  = np.where(~occ_mask)[0][0]

    homo_eV  = float(eig_eV[homo_idx])
    lumo_eV  = float(eig_eV[lumo_idx])
    gap_eV   = lumo_eV - homo_eV

    # Dipole and charges
    dipole_au   = res_iface.get_dipole()          # Debye-ish (actually a.u.)
    charges     = res_iface.get_charges()         # Mulliken

    xtb_orb_data = {
        "eig_eV":   eig_eV,
        "occ":      occ_alpha,
        "homo_idx": homo_idx,
        "lumo_idx": lumo_idx,
        "homo_eV":  homo_eV,
        "lumo_eV":  lumo_eV,
        "gap_eV":   gap_eV,
        "charges":  charges,
        "dipole_au": dipole_au,
    }

    AU_TO_DEBYE = 2.541746473
    dipole_D = np.linalg.norm(dipole_au) * AU_TO_DEBYE

    print("═══ GFN2-xTB Orbital Properties ═══════════════════")
    print(f"  MOs computed  : {len(eig_eV)}")
    print(f"  Occupied MOs  : {occ_mask.sum()}")
    print(f"  Virtual MOs   : {(~occ_mask).sum()}")
    print()
    print(f"  HOMO  (MO #{homo_idx:3d}) : {homo_eV:+.4f} eV")
    print(f"  LUMO  (MO #{lumo_idx:3d}) : {lumo_eV:+.4f} eV")
    print(f"  Gap               : {gap_eV:.4f} eV")
    print()
    print(f"  Dipole moment     : {dipole_D:.4f} D")
    print(f"  Dipole (a.u.)     : [{dipole_au[0]:+.4f}  {dipole_au[1]:+.4f}  {dipole_au[2]:+.4f}]")

In [ ]:
if xtb_orb_data is None:
    print("No xTB orbital data — skipping plot.")
else:
    eig_eV   = xtb_orb_data["eig_eV"]
    occ      = xtb_orb_data["occ"]
    homo_idx = xtb_orb_data["homo_idx"]
    lumo_idx = xtb_orb_data["lumo_idx"]
    homo_eV  = xtb_orb_data["homo_eV"]
    lumo_eV  = xtb_orb_data["lumo_eV"]
    gap_eV   = xtb_orb_data["gap_eV"]

    # Show frontier orbitals ±5 around HOMO/LUMO
    window = 5
    lo = max(0, homo_idx - window)
    hi = min(len(eig_eV), lumo_idx + window + 1)
    frontier_e   = eig_eV[lo:hi]
    frontier_occ = occ[lo:hi]
    frontier_idx = np.arange(lo, hi)

    fig, ax = plt.subplots(figsize=(9, 4))
    for i, (e, o, mo) in enumerate(zip(frontier_e, frontier_occ, frontier_idx)):
        is_homo = mo == homo_idx
        is_lumo = mo == lumo_idx
        color  = "#c62828" if is_homo else ("#1565c0" if is_lumo else
                 ("#555" if o > 0.5 else "#aaa"))
        lw     = 2.5 if (is_homo or is_lumo) else 1.0
        label  = "HOMO" if is_homo else ("LUMO" if is_lumo else None)
        ax.hlines(e, i - 0.35, i + 0.35, colors=color, linewidths=lw, label=label)
        if is_homo or is_lumo:
            ax.text(i + 0.40, e, f"{e:+.3f} eV  ({label})",
                    va="center", fontsize=8, color=color)

    # HOMO-LUMO gap shading
    ax.axhspan(homo_eV, lumo_eV, alpha=0.10, color="#fdd835",
               label=f"Gap = {gap_eV:.3f} eV")

    ax.set_xticks(range(len(frontier_e)))
    ax.set_xticklabels([f"{mo}" for mo in frontier_idx], fontsize=7)
    ax.set_xlabel("MO index")
    ax.set_ylabel("Energy (eV)")
    ax.set_title("H₄DOBDC — Frontier molecular orbital energies (GFN2-xTB)")
    handles = [
        plt.Line2D([0], [0], color="#c62828", lw=2, label=f"HOMO {homo_eV:+.3f} eV"),
        plt.Line2D([0], [0], color="#1565c0", lw=2, label=f"LUMO {lumo_eV:+.3f} eV"),
        plt.Rectangle([0,0], 1, 1, fc="#fdd835", alpha=0.5,
                      label=f"Gap = {gap_eV:.3f} eV"),
    ]
    ax.legend(handles=handles, loc="lower right")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()

In [ ]:
if xtb_orb_data is not None:
    charges = xtb_orb_data["charges"]
    syms    = atoms_qm.get_chemical_symbols()

    fig, ax = plt.subplots(figsize=(9, 3))
    colors = ["#cc2200" if q < 0 else "#1565c0" for q in charges]
    bars = ax.bar(range(len(charges)), charges, color=colors, width=0.7)
    ax.set_xticks(range(len(charges)))
    ax.set_xticklabels([f"{i}\n{s}" for i, s in enumerate(syms)], fontsize=7)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("Mulliken charge (e)")
    ax.set_title("H₄DOBDC — GFN2-xTB Mulliken partial charges")
    ax.set_xlabel("Atom index / element")
    fig.tight_layout()
    plt.show()

    # Print heavy-atom charges
    print("Heavy-atom charges:")
    for i, (s, q) in enumerate(zip(syms, charges)):
        if s != "H":
            print(f"  atom {i:2d}  {s}  : {q:+.4f} e")

---
## 7 · ORCA Electronic Properties

ORCA provides DFT-quality HOMO/LUMO energies. Two setups are available:

| Option | Keywords | Speed | Quality |
|---|---|---|---|
| `USE_XTB2_IN_ORCA = True` | `! XTB2` | Fast (seconds) | GFN2-xTB via ORCA |
| `USE_XTB2_IN_ORCA = False` | `! B3LYP def2-SVP D3BJ TightSCF` | ~minutes | Good DFT |

**Set `ORCA_CMD` in the config cell** to enable this section.  
The helper `parse_orca_orbital_energies()` reads HOMO/LUMO from the ORCA `.out` file.

> For high-throughput, wrap in a shell loop.  The ASE calculator is reusable.

In [ ]:
def parse_orca_orbital_energies(orca_out_path: str | Path) -> dict:
    """Extract HOMO/LUMO/gap from an ORCA output file.

    Reads the 'ORBITAL ENERGIES' block that ORCA prints after every SCF.
    Returns a dict with homo_eV, lumo_eV, gap_eV, and the full MO table.

    Works for both restricted (RHF/RKS) and unrestricted (UHF/UKS) outputs;
    for the latter, only the alpha-spin channel is returned.
    """
    text = Path(orca_out_path).read_text(errors="replace")

    # ORCA prints zero or more ORBITAL ENERGIES blocks; the last one is final.
    # Pattern: header, dash line, blank, column header, then rows of numbers.
    block_re = re.compile(
        r"ORBITAL ENERGIES\s*\n\s*-+\s*\n"
        r"(?:.*?\n)?"          # optional blank / alpha-spin header
        r"\s*NO\s+OCC\s+E\(Eh\)\s+E\(eV\)\s*\n"
        r"((?:\s*\d+.*\n)*)",
        re.IGNORECASE,
    )
    matches = list(block_re.finditer(text))
    if not matches:
        raise ValueError("No ORBITAL ENERGIES block found in ORCA output.")

    # Use the last alpha-spin block
    orb_block = matches[-1].group(1)

    occupied, virtual = [], []
    for line in orb_block.splitlines():
        parts = line.split()
        if len(parts) < 4:
            continue
        try:
            no    = int(parts[0])
            occ   = float(parts[1])
            e_eh  = float(parts[2])
            e_ev  = float(parts[3])
        except (ValueError, IndexError):
            continue
        row = (no, occ, e_eh, e_ev)
        (occupied if occ > 0.5 else virtual).append(row)

    if not occupied or not virtual:
        raise ValueError("Could not parse orbital rows from ORCA output.")

    homo_no, _, homo_eh, homo_ev = occupied[-1]
    lumo_no, _, lumo_eh, lumo_ev = virtual[0]

    return {
        "homo_idx":  homo_no,
        "lumo_idx":  lumo_no,
        "homo_eV":   homo_ev,
        "lumo_eV":   lumo_ev,
        "gap_eV":    lumo_ev - homo_ev,
        "homo_Eh":   homo_eh,
        "lumo_Eh":   lumo_eh,
        "occupied":  occupied,
        "virtual":   virtual,
    }

print("parse_orca_orbital_energies() defined.")

In [ ]:
orca_data = None

if ORCA_CMD is None:
    print("ORCA_CMD is None — skipping ORCA calculation.")
    print("Set ORCA_CMD in the config cell to enable this section.")

elif not HAS_ORCA_ASE:
    print("ASE ORCA interface not found (check ASE installation).")

elif not Path(ORCA_CMD).exists():
    print(f"ORCA binary not found at: {ORCA_CMD}")
    print("Update ORCA_CMD in the config cell.")

else:
    orca_dir = WORK_DIR / "orca_run"
    orca_dir.mkdir(exist_ok=True)

    kw = ORCA_XTB2_KEYWORDS if USE_XTB2_IN_ORCA else ORCA_KEYWORDS
    print(f"Running ORCA:  {kw}")
    print(f"Directory   :  {orca_dir.resolve()}")

    calc_orca = ORCA(
        profile=OrcaProfile(command=ORCA_CMD),
        template=OrcaTemplate(
            keywords=kw,
            charge=CHARGE,
            multiplicity=UHF + 1,    # 2S+1, singlet = 1
        ),
        directory=str(orca_dir),
    )

    atoms_orca = atoms_qm.copy()
    atoms_orca.calc = calc_orca

    # Run energy/gradient
    e_orca = atoms_orca.get_potential_energy()
    print(f"ORCA energy : {e_orca:.6f} eV")

    # Parse HOMO/LUMO from output file
    orca_out = orca_dir / "orca.out"
    if not orca_out.exists():
        # Older ASE versions may use a different filename
        candidates = list(orca_dir.glob("*.out"))
        orca_out = candidates[0] if candidates else None

    if orca_out and orca_out.exists():
        orca_data = parse_orca_orbital_energies(orca_out)
        orca_data["energy_eV"] = e_orca
        orca_data["method"]    = "XTB2" if USE_XTB2_IN_ORCA else "B3LYP/def2-SVP"

        print()
        print("═══ ORCA Orbital Properties ═══════════════════════")
        print(f"  Method        : {orca_data['method']}")
        print(f"  HOMO (MO #{orca_data['homo_idx']:3d}) : {orca_data['homo_eV']:+.4f} eV")
        print(f"  LUMO (MO #{orca_data['lumo_idx']:3d}) : {orca_data['lumo_eV']:+.4f} eV")
        print(f"  Gap           : {orca_data['gap_eV']:.4f} eV")
    else:
        print(f"⚠  Could not find ORCA output file in {orca_dir}")

In [ ]:
if orca_data is None:
    print("No ORCA data available — skipping orbital energy plot.")
else:
    occupied = orca_data["occupied"]
    virtual  = orca_data["virtual"]

    # Frontier: last 6 occupied, first 6 virtual
    frontier = occupied[-6:] + virtual[:6]
    n_occ_f  = 6

    fig, ax = plt.subplots(figsize=(10, 4))
    for i, (no, occ, e_eh, e_ev) in enumerate(frontier):
        is_homo = no == orca_data["homo_idx"]
        is_lumo = no == orca_data["lumo_idx"]
        color = "#c62828" if is_homo else ("#1565c0" if is_lumo else
                ("#555" if occ > 0.5 else "#aaa"))
        lw = 2.5 if (is_homo or is_lumo) else 1.2
        ax.hlines(e_ev, i - 0.35, i + 0.35, colors=color, linewidths=lw)
        if is_homo or is_lumo:
            label = "HOMO" if is_homo else "LUMO"
            ax.text(i + 0.40, e_ev, f"{e_ev:+.3f} eV  ({label})",
                    va="center", fontsize=8, color=color)

    ax.axhspan(orca_data["homo_eV"], orca_data["lumo_eV"],
               alpha=0.10, color="#fdd835")
    ax.set_xticks(range(len(frontier)))
    ax.set_xticklabels([f"{row[0]}" for row in frontier], fontsize=7)
    ax.set_xlabel("MO index")
    ax.set_ylabel("Energy (eV)")
    ax.set_title(f"H₄DOBDC — Frontier MO energies ({orca_data['method']})")
    ax.grid(axis="y", alpha=0.25)

    handles = [
        plt.Line2D([0],[0], color="#c62828", lw=2,
                   label=f"HOMO {orca_data['homo_eV']:+.3f} eV"),
        plt.Line2D([0],[0], color="#1565c0", lw=2,
                   label=f"LUMO {orca_data['lumo_eV']:+.3f} eV"),
        plt.Rectangle([0,0],1,1, fc="#fdd835", alpha=0.5,
                      label=f"Gap = {orca_data['gap_eV']:.3f} eV"),
    ]
    ax.legend(handles=handles, loc="lower right")
    fig.tight_layout()
    plt.show()

---
## 8 · Conceptual DFT Descriptors

Koopmans' theorem links frontier orbital energies to chemical reactivity:

| Descriptor | Formula | Meaning |
|---|---|---|
| Ionisation potential (IP) | $-\varepsilon_{\text{HOMO}}$ | Cost to remove an electron |
| Electron affinity (EA) | $-\varepsilon_{\text{LUMO}}$ | Gain from adding an electron |
| Chemical hardness | $\eta = (IP - EA)/2 = \Delta E_{\text{gap}}/2$ | Resistance to charge transfer |
| Chemical potential | $\mu = -(IP + EA)/2$ | Tendency to donate electrons |
| Electronegativity | $\chi = -\mu$ | Pauling-scale analogue |
| Electrophilicity | $\omega = \mu^2/(2\eta)$ | Electrophilic reactivity |

These are widely used to rationalise MOF linker–metal binding and photocatalytic properties.

In [ ]:
def compute_cdft(homo_eV: float, lumo_eV: float, label: str = "") -> dict:
    """Compute conceptual-DFT reactivity descriptors from HOMO/LUMO energies."""
    ip   = -homo_eV          # ionisation potential (Koopmans)
    ea   = -lumo_eV          # electron affinity
    gap  = lumo_eV - homo_eV
    eta  = gap / 2.0          # chemical hardness
    mu   = (homo_eV + lumo_eV) / 2.0   # chemical potential
    chi  = -mu                # electronegativity
    omega = mu**2 / (2 * eta) if eta > 1e-10 else float("nan")  # electrophilicity

    return {
        "label":    label,
        "HOMO_eV":  homo_eV,
        "LUMO_eV":  lumo_eV,
        "gap_eV":   gap,
        "IP_eV":    ip,
        "EA_eV":    ea,
        "eta_eV":   eta,
        "mu_eV":    mu,
        "chi_eV":   chi,
        "omega_eV": omega,
    }

results_table = []

if xtb_orb_data is not None:
    results_table.append(
        compute_cdft(xtb_orb_data["homo_eV"], xtb_orb_data["lumo_eV"], label="GFN2-xTB")
    )

if orca_data is not None:
    results_table.append(
        compute_cdft(orca_data["homo_eV"], orca_data["lumo_eV"], label=orca_data["method"])
    )

if results_table:
    print("═══ Conceptual DFT Descriptors for H₄DOBDC (neutral acid) ═══════════")
    header = f"{'Descriptor':<28}" + "  ".join(f"{r['label']:>14}" for r in results_table)
    print(header)
    print("-" * len(header))
    rows = [
        ("HOMO (eV)",      "homo_eV"),
        ("LUMO (eV)",      "lumo_eV"),
        ("Gap  Δε (eV)",   "gap_eV"),
        ("IP   −ε_HOMO",   "IP_eV"),
        ("EA   −ε_LUMO",   "EA_eV"),
        ("η    hardness",  "eta_eV"),
        ("μ    chem. pot.","mu_eV"),
        ("χ    electronegativity", "chi_eV"),
        ("ω    electrophilicity",  "omega_eV"),
    ]
    for name, key in rows:
        vals = "  ".join(f"{r[key]:+14.4f}" for r in results_table)
        print(f"  {name:<26}{vals}")
    print()
    print("Units: all values in eV.")
else:
    print("No orbital data available — run xTB and/or ORCA sections first.")

In [ ]:
if not results_table:
    print("No data to plot.")
else:
    labels   = [r["label"] for r in results_table]
    descriptors = ["IP_eV", "EA_eV", "gap_eV", "eta_eV", "chi_eV", "omega_eV"]
    desc_names  = ["IP (eV)", "EA (eV)", "Gap (eV)", "η hard. (eV)",
                   "χ (eV)", "ω electr. (eV)"]

    x  = np.arange(len(descriptors))
    w  = 0.8 / max(len(results_table), 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    colors_bar = ["#1565c0", "#c62828", "#2e7d32", "#6a1b9a"]
    for i, (r, col) in enumerate(zip(results_table, colors_bar)):
        vals = [abs(r[k]) for k in descriptors]   # plot magnitudes
        offset = (i - (len(results_table) - 1) / 2) * w
        ax.bar(x + offset, vals, width=w * 0.9, color=col, alpha=0.80,
               label=r["label"])

    ax.set_xticks(x)
    ax.set_xticklabels(desc_names)
    ax.set_ylabel("Value (eV)")
    ax.set_title("H₄DOBDC — Conceptual DFT reactivity descriptors")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()

---
## 9 · Summary

In [ ]:
print("══════════════════════════════════════════════════════════════")
print("  H₄DOBDC (2,5-dihydroxyterephthalic acid) — Calculation Summary")
print("══════════════════════════════════════════════════════════════")
print(f"  SMILES       : {SMILES}")
print(f"  Formula      : {atoms_qm.get_chemical_formula()}")
print(f"  Atoms        : {len(atoms_qm)}")
print(f"  Charge/UHF   : {CHARGE} / {UHF}")
print()

print("  ── Geometry ──────────────────────────────────────────────")
print(f"  Source       : SMILES → RDKit ETKDG + MMFF94")
if HAS_MACE:
    print(f"  Relaxed by   : MACE-MP-0 ({MACE_MODEL}) on {MACE_DEVICE}")
    print(f"  MACE energy  : {energy_mace:.4f} eV")
print()

print("  ── GOAL AtomicGraph ───────────────────────────────────────")
print(f"  Cutoff       : {CUTOFF} Å")
print(f"  Nodes N      : {graph.num_atoms}")
print(f"  Edges E      : {graph.edge_index.shape[1]}")
print(f"  Avg degree   : {graph.edge_index.shape[1] / graph.num_atoms:.2f}")
print()

if results_table:
    print("  ── Electronic Properties ──────────────────────────────────")
    for r in results_table:
        print(f"  Method  : {r['label']}")
        print(f"    HOMO  : {r['homo_eV']:+.4f} eV")
        print(f"    LUMO  : {r['lumo_eV']:+.4f} eV")
        print(f"    Gap   : {r['gap_eV']:.4f} eV")
        print(f"    IP    : {r['IP_eV']:.4f} eV  (Koopmans)")
        print(f"    EA    : {r['EA_eV']:.4f} eV  (Koopmans)")
        print(f"    η     : {r['eta_eV']:.4f} eV  (chemical hardness)")
        print(f"    μ     : {r['mu_eV']:+.4f} eV  (chemical potential)")
        print(f"    ω     : {r['omega_eV']:.4f} eV  (electrophilicity)")
        print()

print("  ── Output files ────────────────────────────────────────────")
for f in sorted(WORK_DIR.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(WORK_DIR)}  ({size_kb:.1f} kB)")
print("══════════════════════════════════════════════════════════════")

---
## Next Steps

| Goal | How |
|---|---|
| **Deprotonated ligand** | Change `CHARGE = -4` and rerun all cells |
| **Mn coordination complex** | Add Mn atoms, set `CHARGE = 0` or `+2`, use ORCA with larger basis |
| **Full MOF cluster model** | Extract a Mn₃(DOBDC)₃ cluster from a CIF; load with `ase.io.read()` |
| **Swap MACE → SIMURGH** | Replace `calc_mace` with `GOALCalculator(checkpoint_path=...)` |
| **Higher DFT** | Change `ORCA_KEYWORDS` to `! wB97M-V def2-TZVP RIJCOSX TightSCF` |
| **UV-Vis / excited states** | Add `! TD-DFT` block in ORCA template |
| **NBO / QTAIM** | Use ORCA's NBO interface or Multiwfn on the .gbw file |